# RS02 · MISRA C, static analysis, traceability and IEC 62304

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BrandonZabawa/robotics-daily-labs/blob/main/labs/07_reliability_systems/RS02_misra_c_static_analysis_traceability_and_iec_623.ipynb)

**Track:** Reliability and systems · **Skill axis (sheet):** Reliability and systems · **Time:** about 55 min

**Skill Hub concepts:** MISRA C and static analysis; Requirements traceability; IEC 62304 software safety classes

**Level 1** (the base exercises). Harder levels go at the end of this notebook.

> Cold attempt rules: notes closed, no AI, no internet until you run `lab.score()`. Fill every `TODO`, run the check under it, then do the checkpoint.

In [ ]:
#@title ⚙️ Setup: run this cell first
import os, sys, pathlib, subprocess
REPO_URL = "https://github.com/BrandonZabawa/robotics-daily-labs"   # or your fork
BRANCH = "main"                                  # the branch that holds your added levels
_root = os.environ.get("RDL_ROOT")
_root = pathlib.Path(_root) if _root else next((p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "roboprep").is_dir()), None)
if _root is None:
    _root = pathlib.Path("/content/robotics-daily-labs")
    if _root.exists():
        subprocess.run(["git", "-C", str(_root), "fetch", "-q", "origin", BRANCH])
        subprocess.run(["git", "-C", str(_root), "checkout", "-q", BRANCH])
        subprocess.run(["git", "-C", str(_root), "pull", "-q", "origin", BRANCH])
    else:
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", BRANCH, REPO_URL, str(_root)], check=True)
if str(_root) not in sys.path:
    sys.path.insert(0, str(_root))
from roboprep import Lab
lab = Lab("RS02")

## Map: how safety-related firmware proves it is what it claims

```
 requirement REQ-3 "stop the motor within 10 ms of over-current"
       └──> design: motor.c / motor_step()        └──> test: test_overcurrent_stops()  /* @verifies REQ-3 */
 traceability matrix: every requirement has at least one passing test, every test points at a requirement
```

| MISRA C:2012 rule (examples) | Says | Why |
|---|---|---|
| 13.4 | do not use the result of an assignment (`if ((y = f()) > 0)`) | easily confused with `==` |
| 15.1 | no `goto` | unstructured control flow |
| 15.7 | every `if ... else if` chain ends with an `else` | forces you to handle "none of the above" |
| 16.4 | every `switch` has a `default` | same, for switches |
| 17.7 | use (or cast to `void`) every non-void return value | ignoring an error code is usually a bug |
| 10.x | essential type model: no implicit mixing of signed/unsigned, narrowing, bool/int | lab C02 |

Static analyzers: cppcheck (with its MISRA addon), clang-tidy, gcc `-fanalyzer`, commercial (Polyspace, PC-lint,
Coverity). A **deviation** documents why a rule is broken on purpose. BARR-C is a lighter style standard aimed at bugs.

| IEC 62304 software safety class | If the software fails, the worst outcome is | Rigor |
|---|---|---|
| A | no injury or damage to health possible | basic process |
| B | non-serious injury | plus architecture and unit verification |
| C | death or serious injury | plus detailed design, unit-level testing of everything |

In [ ]:
from roboprep import clab
clab.ensure_apt("cppcheck", "cppcheck")

## Task 1: make this function MISRA-clean without changing what it does
Behavior: `y = x / 2`, and if `y > 10` then `last = y`; then if `x` is 0 or 1, `last = x`; then if `x > 5` return `last`
immediately, otherwise call `read_sensor()` (its result is not needed) and return `last`.
The checker runs cppcheck's MISRA addon (rules 13.4, 15.1, 15.7, 16.4, 17.7 must be gone) and a behavior test.

In [ ]:
%%writefile filter.c
#include <stdint.h>
int32_t read_sensor(void);
int32_t filter(int32_t x);
static int32_t last;

int32_t filter(int32_t x) {
    int32_t y;
    if ((y = x / 2) > 10) { last = y; }
    switch (x) {
    case 0: last = 0; break;
    case 1: last = 1; break;
    }
    if (x > 5) { goto out; }
    read_sensor();
    out:
    return last;
}

In [ ]:
print(clab.sh("cppcheck -q --addon=misra --enable=style filter.c 2>&1 | grep -o 'misra-c2012-[0-9.]*' | sort | uniq -c").stdout or "no MISRA findings")
lab.check("misra")

## Task 2: build the traceability matrix
Tests are C functions named `test_*`; a comment `/* @verifies REQ-n */` (one or more IDs, comma-separated) sits on
the line right above each one. Return `{req_id: [test names]}` for every requirement (empty list if untested).

In [ ]:
import re

def trace(req_ids, test_source):
    raise NotImplementedError  # TODO

lab.check("trace", trace)

## Checkpoint: Math
**M1 (Apply).** 18 of 20 requirements have at least one passing test. Requirement coverage in percent?

**M2 (Apply).** cppcheck reports 12 findings; 3 are justified deviations you document. How many must you fix?

In [ ]:
lab.answer("M1", None)
lab.answer("M2", None)

## Checkpoint: Concept
**K1 (Apply).** Surgical robot arm firmware whose failure could move the tool into the patient. IEC 62304 class?
- a) A
- b) C (it could contribute to death or serious injury), unless hardware measures outside the software reduce the risk
- c) B

**K2 (Evaluate).** Your team wants to adopt MISRA C on the Storm firmware. Propose how: which rules first, how tooling
and deviations work, and how to avoid drowning in findings on generated vendor code.

In [ ]:
lab.choose("K1", None)

In [ ]:
lab.explain("K2", """
Your explanation here.
""")

In [ ]:
lab.self_score({"K2": 0})

## Level 2 and beyond: harder exercises you add

Add each new exercise above the score cell, in this order:

1. A markdown cell: what to build and why (a table or diagram if it helps).
2. A code cell with your stub (Python, or a `%%writefile` C/C++ cell that includes `rs02.h`).
3. In `roboprep/checks/rs02.py`, a grader: `def check_L2_<name>(fn): assert ...` (for C/C++ also
   `def c_L2_<name>(lib): return (callable, ...)` built with `roboprep.cbind.Functions`). Every `check_` function in that
   file counts toward the code score automatically.
4. `lab.check("L2_<name>", your_function)` (or the C/C++ file name) under your stub.

Commit the notebook and the grader to your branch and set `BRANCH` in the setup cell so Colab clones it.

In [ ]:
# Level 2: your exercises go here (see the cell above)

In [ ]:
lab.score()